# Day 2 — 추론의 원리: KV Cache와 TTFT/TPOT

Day 1에서 `model.generate()`로 텍스트를 생성해봤습니다. 오늘은 그 안에서 실제로 무슨 일이 일어나는지, 왜 LLM 추론이 느린지, 그리고 그 느림을 어떻게 정량화하는지(TTFT/TPOT) 다룹니다.

이번에도 Day 1에서 쓴 `Qwen/Qwen2.5-0.5B-Instruct`를 그대로 재사용합니다 — 새 개념에 집중하기 위해 모델은 고정합니다.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16).to("cuda")
model.eval()

print("모델 로드 완료:", model_name)
print("파라미터 수:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

## 1. Autoregressive 생성은 왜 느린가

LLM은 토큰을 한 번에 하나씩만 생성합니다 (`t_1 → t_2 → t_3 → ...`). 각 토큰을 만들 때 모델은 **지금까지의 전체 시퀀스**를 입력받아 self-attention을 계산합니다.

순진하게 구현하면: n번째 토큰을 생성할 때 길이 n짜리 시퀀스 전체를 처음부터 다시 forward pass 해야 합니다. 즉 총 연산량이 1 + 2 + 3 + ... + N ≈ O(N²) 로 늘어납니다 — 매 스텝마다 이전에 이미 계산했던 토큰들의 attention key/value를 **또 계산**하기 때문입니다.

이 중복 계산을 없애는 것이 바로 **KV Cache**입니다: 각 레이어의 attention key/value 텐서를 한 번 계산하면 재사용을 위해 저장해두고, 다음 스텝에서는 새로 생긴 토큰 1개에 대해서만 key/value를 계산해 캐시에 이어붙입니다. 그 결과 스텝당 연산량이 O(1)에 가깝게 줄고, 전체 생성 연산량은 O(N)이 됩니다.

> **직접 해보고 알게 된 것**: `model.generate(..., use_cache=False)`로 끄면 될 것 같지만, 이 환경(transformers 5.13.0)에서 실제로 측정해보면 `use_cache=False`를 줘도 속도가 거의 그대로입니다 — `generate()`가 내부적으로 이 플래그를 무시하는 것으로 보입니다. 그래서 아래에서는 `generate()`에 의존하지 않고, `model()`을 직접 반복 호출하는 **수동 루프**로 "캐시 있음"과 "매 스텝 전체 재계산"을 직접 구현해서 비교합니다. 이렇게 하면 정확히 무엇이 재사용되고 무엇이 재계산되는지 제어할 수 있습니다.

먼저 두 가지 방식을 직접 구현합니다:
- `manual_with_cache`: prefill 1번 + 이후 매 스텝 새 토큰 1개만 forward (캐시 재사용)
- `manual_no_cache`: 매 스텝마다 지금까지의 전체 시퀀스를 처음부터 다시 forward (캐시 없음)

In [ ]:
import time

messages = [{"role": "user", "content": "인공지능의 역사에 대해 설명해줘"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 워밍업: Day1에서 본 CC8.7 PTX JIT 컴파일 비용을 여기서 미리 태워버린다.
# (이걸 안 하면 아래 비교 실험의 첫 결과가 'JIT 컴파일 시간'까지 섞여서 왜곡된다)
with torch.no_grad():
    _ = model.generate(**inputs, max_new_tokens=5, use_cache=True)
torch.cuda.synchronize()
print("워밍업 완료")

In [ ]:
def manual_with_cache(n_new):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model(**inputs, use_cache=True)
    past = out.past_key_values
    cur = out.logits[:, -1, :].argmax(-1, keepdim=True)
    for _ in range(n_new - 1):
        with torch.no_grad():
            out = model(input_ids=cur, past_key_values=past, use_cache=True)
        past = out.past_key_values
        cur = out.logits[:, -1, :].argmax(-1, keepdim=True)
    torch.cuda.synchronize()
    return time.perf_counter() - t0


def manual_no_cache(n_new):
    seq = inputs["input_ids"]
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_new):
        with torch.no_grad():
            out = model(input_ids=seq, use_cache=False)
        nxt = out.logits[:, -1, :].argmax(-1, keepdim=True)
        seq = torch.cat([seq, nxt], dim=1)
    torch.cuda.synchronize()
    return time.perf_counter() - t0


# 주의: n_new를 너무 크게(예: 500+) 잡으면 manual_no_cache가 매 스텝 점점 길어지는
# 시퀀스 전체를 다시 forward하면서 8GB 통합 메모리를 다 써버려 OOM으로 프로세스가 죽을 수
# 있습니다 (실제로 겪었습니다). 150 이하로 유지하세요.
for n in [30, 80, 150]:
    tc = manual_with_cache(n)
    tnc = manual_no_cache(n)
    print(f"n={n:4d} | with-cache: {tc:.3f}s | no-cache(전체 재계산): {tnc:.3f}s | 배율: {tnc/tc:.2f}x")

**실측 결과** (이 하드웨어, Qwen2.5-0.5B-Instruct 기준):

```
n=  30 | with-cache: 2.37s | no-cache: 2.50s | 배율: 1.05x
n=  80 | with-cache: 6.31s | no-cache: 6.66s | 배율: 1.06x
n= 150 | with-cache: 11.83s | no-cache: 13.39s | 배율: 1.13x
```

**정직하게 짚을 점**: 배율이 "극적으로" 크지 않습니다. 이유는 두 가지입니다:

1. **모델이 작다(0.5B)**: attention 재계산 자체의 연산량이 작아서, Python 루프/커널 실행(launch) 같은 고정 오버헤드에 비해 상대적으로 작은 비중을 차지합니다. 7B급 이상 모델이나 시퀀스 길이가 수천 토큰 단위로 커지면 이 배율은 훨씬 커집니다 (이게 실무에서 KV cache 없이는 대형 모델 서빙이 사실상 불가능한 이유입니다).
2. **시퀀스 길이가 여전히 짧다(30~150 토큰)**: O(N²) vs O(N)의 차이는 N이 커질수록 벌어집니다. n을 더 키워서 확인하고 싶겠지만, 이 노트북에서 n=500으로 시도했을 때 `manual_no_cache`가 8GB 통합 메모리를 다 소진해 프로세스가 강제 종료(OOM kill)되는 것을 실제로 확인했습니다 — 그래서 안전하게 150 이하로 제한했습니다.

그래도 추세는 명확합니다: n이 30→80→150으로 커질수록 배율이 1.05x→1.06x→1.13x로 **꾸준히 증가**합니다. 이 증가 추세 자체가 O(N²) 대 O(N)의 격차가 실제로 벌어지고 있다는 증거입니다. 실무에서 쓰이는 7B~70B급 모델, 수천 토큰 컨텍스트에서는 이 격차가 수십~수백 배로 벌어지기 때문에, KV cache는 '있으면 좋은 최적화'가 아니라 '없으면 실사용이 불가능한 필수 장치'입니다.

## 2. KV Cache 내부 들여다보기: `past_key_values`

`generate()`는 내부적으로 `model.forward()`를 반복 호출하면서 `past_key_values`(캐시)를 다음 호출에 넘겨줍니다. 이걸 한 스텝씩 직접 재현해서 캐시가 실제로 무엇을 담고 있는지 확인해봅시다.

In [ ]:
with torch.no_grad():
    # 1스텝: prompt 전체를 한 번에 forward (= prefill)
    out = model(**inputs, use_cache=True)

kv_cache = out.past_key_values
print("레이어 수:", len(kv_cache))

# 최신 transformers는 past_key_values를 Cache 객체로 반환한다.
# 레이어별 key 텐서 shape로 확인: (batch, num_heads, seq_len, head_dim)
first_layer_key = kv_cache.layers[0].keys
print("레이어0 key shape:", first_layer_key.shape)
print("-> seq_len 차원이 지금까지 처리한 토큰 수(prompt 길이)와 같다:", inputs["input_ids"].shape[1])

In [ ]:
# 다음 토큰 하나를 뽑아서, 그 토큰 '하나만' 다시 forward에 넣고 캐시를 이어붙여본다.
next_token_logits = out.logits[:, -1, :]
next_token_id = next_token_logits.argmax(dim=-1, keepdim=True)
print("방금 생성된 토큰:", tokenizer.decode(next_token_id[0]))

with torch.no_grad():
    out2 = model(input_ids=next_token_id, past_key_values=kv_cache, use_cache=True)

new_key_len = out2.past_key_values.layers[0].keys.shape[2]
print("캐시 업데이트 후 레이어0 key shape의 seq_len:", new_key_len)
print("-> prompt 길이 + 1 (새 토큰 1개만 계산해서 이어붙임, 전체 재계산 아님)")

**핵심**: 두 번째 forward 호출은 입력으로 토큰 **1개**만 넣었습니다 (`next_token_id`, shape `(1, 1)`). 그런데도 결과 캐시의 seq_len은 prompt 길이 + 1이 되었습니다. 이게 KV cache의 정확한 동작입니다 — 새 토큰에 대한 key/value만 계산해서 기존 캐시 뒤에 이어붙일 뿐, 이전 토큰들의 key/value는 절대 다시 계산하지 않습니다.

이 구조 때문에 생성 과정은 자연스럽게 두 단계로 나뉩니다:

- **Prefill**: prompt 전체를 한 번에 forward → 캐시를 채움 (병렬 처리 가능, GPU를 꽉 채워 씀)
- **Decode**: 캐시를 재사용하며 토큰을 하나씩 순차 생성 (매 스텝이 순차적이라 병렬화가 안 됨)

다음 섹션에서 이 두 단계를 각각 시간으로 측정하는 지표, TTFT와 TPOT를 다룹니다.

## 3. TTFT / TPOT 정의

LLM 서빙에서 '생성이 얼마나 걸리는가'는 보통 총 시간 하나로 뭉뚱그리지 않고 두 지표로 쪼개서 봅니다.

- **TTFT (Time To First Token)**: 요청이 들어온 시점부터 첫 번째 출력 토큰이 나오기까지 걸리는 시간. 이 구간이 바로 위에서 본 **prefill**에 해당합니다 — prompt 전체를 처리하는 데 걸리는 시간이므로, prompt가 길수록 TTFT가 커집니다.
- **TPOT (Time Per Output Token)**: 첫 토큰 이후, 토큰 하나를 더 생성하는 데 평균적으로 걸리는 시간. **decode** 단계의 스텝당 비용이며, KV cache 덕분에 이론상 prompt 길이와 거의 무관하게 일정합니다 (생성이 길어질수록 캐시가 커지므로 아주 약간씩 늘어나긴 합니다).

왜 나눠서 보나: 챗봇처럼 '얼마나 빨리 응답이 시작되는지'가 중요한 경우 TTFT가 체감 속도를 좌우하고, 긴 답변을 끝까지 읽는 경우엔 TPOT가 전체 완료 시간을 좌우합니다. 두 지표는 서로 다른 병목(prefill의 연산량 vs decode의 순차성)에서 나오기 때문에 최적화 방법도 다릅니다 (예: prefill은 배치/병렬화로, decode는 KV cache 메모리 대역폭으로 병목).

In [ ]:
def measure_ttft_tpot(prompt_text, max_new_tokens=40):
    msgs = [{"role": "user", "content": prompt_text}]
    formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(formatted, return_tensors="pt").to("cuda")

    # --- TTFT: prompt 전체 prefill + 첫 토큰 산출 ---
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model(**inp, use_cache=True)
    next_logits = out.logits[:, -1, :]
    next_id = next_logits.argmax(dim=-1, keepdim=True)
    torch.cuda.synchronize()
    ttft = time.perf_counter() - t0

    # --- TPOT: 이후 토큰들을 하나씩 순차 생성, 평균 스텝 시간 ---
    past = out.past_key_values
    generated = [next_id]
    step_times = []
    cur_input = next_id
    for _ in range(max_new_tokens - 1):
        torch.cuda.synchronize()
        ts = time.perf_counter()
        with torch.no_grad():
            step_out = model(input_ids=cur_input, past_key_values=past, use_cache=True)
        torch.cuda.synchronize()
        step_times.append(time.perf_counter() - ts)

        past = step_out.past_key_values
        cur_input = step_out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated.append(cur_input)

    tpot = sum(step_times) / len(step_times)
    prompt_len = inp["input_ids"].shape[1]
    return ttft, tpot, prompt_len

# 콜드 상태 없이(이미 워밍업 했으므로) 측정
ttft, tpot, plen = measure_ttft_tpot("인공지능의 역사에 대해 설명해줘")
print(f"prompt 길이: {plen} 토큰")
print(f"TTFT: {ttft*1000:.1f} ms")
print(f"TPOT: {tpot*1000:.1f} ms/token  (-> {1/tpot:.2f} tok/s)")

## 4. 콜드스타트가 TTFT 측정을 어떻게 오염시키는가

Day 1에서 첫 생성이 ~2 tok/s, 워밍업 후엔 ~8 tok/s였던 걸 봤습니다. 그 원인이 CC 8.7용 사전 컴파일 커널이 없어 PTX를 JIT 컴파일하는 비용이었죠. 이 비용은 정확히 **TTFT를 부풀리는 방식**으로 나타납니다 — JIT 컴파일은 커널이 처음 실행될 때 한 번만 발생하고, 그 커널들은 대부분 prefill의 첫 forward 호출에서 이미 다 실행되기 때문입니다.

즉 '콜드스타트 지연'은 TPOT가 아니라 거의 전부 TTFT에 실린다는 뜻입니다. 방금 위에서는 이미 워밍업된 모델로 측정했으니 이번엔 프로세스를 다시 시작한 것처럼 재현할 수는 없지만, 같은 효과를 **다른 종류의 최초 실행**으로 재현해볼 수 있습니다: 지금까지 한 번도 실행 안 해본 배치 크기/시퀀스 길이 조합은 새로운 커널 디스패치 경로를 타면서 다시 소폭의 JIT 비용이 붙습니다.

In [ ]:
# 지금까지 써보지 않은 조합: 훨씬 긴 prompt (다른 shape의 attention 커널 경로를 처음 탈 수 있음)
long_prompt = "다음 주제들을 각각 두 문장으로 요약해줘: " + ", ".join(
    [f"주제{i}" for i in range(1, 60)]
)

ttft_long_first, tpot_long_first, plen_long = measure_ttft_tpot(long_prompt, max_new_tokens=20)
print(f"[처음 보는 긴 prompt] 길이: {plen_long} 토큰")
print(f"  TTFT (1회차): {ttft_long_first*1000:.1f} ms")

# 같은 길이의 요청을 한 번 더 (이번엔 커널 경로가 이미 워밍업됨)
ttft_long_second, tpot_long_second, _ = measure_ttft_tpot(long_prompt, max_new_tokens=20)
print(f"  TTFT (2회차, 재실행): {ttft_long_second*1000:.1f} ms")
print(f"\nTPOT는 두 경우 비슷: {tpot_long_first*1000:.1f} ms vs {tpot_long_second*1000:.1f} ms")
print("-> 새 shape을 처음 만났을 때의 추가 비용도 TTFT(=prefill)쪽에 실리고, TPOT(=decode 스텝)는 상대적으로 안정적이다.")

**해석**: 1회차와 2회차의 TTFT 차이(있다면)는 새로운 시퀀스 길이에 대한 attention 커널 디스패치/컴파일 캐시 워밍업 비용입니다. Jetson처럼 CC별 사전 컴파일 커널이 없는 환경에서는 이 비용이 x86+데이터센터 GPU 대비 상대적으로 크게 나타날 수 있습니다.

**실무 교훈**: 이 하드웨어에서 TTFT를 벤치마크할 때는 반드시 (1) 모델 로드 직후 더미 요청으로 워밍업을 한 번 거치고, (2) 실측하려는 prompt 길이/배치 크기 조합으로 최소 1회 이상 '버리는 실행'을 한 뒤에 측정해야 합니다. 그렇지 않으면 'TTFT가 크다'는 결론이 실제로는 'JIT 컴파일이 안 끝났다'는 뜻일 수 있고, 이는 실제 서비스에서 사용자가 매번 겪는 지연이 아니라 프로세스 최초 구동 시 딱 한 번 겪는 지연입니다.

## 5. Prompt 길이 vs 생성 길이: TTFT/TPOT에 미치는 영향 비교

이론상 TTFT는 prompt 길이(prefill 연산량)에 비례해 커지고, TPOT는 prompt 길이와 거의 무관해야 합니다. 짧은 prompt와 긴 prompt로 직접 비교해봅니다.

In [ ]:
short_prompt = "안녕"
mid_prompt = "딥러닝에서 attention 메커니즘이 왜 중요한지 설명해줘"
long_prompt2 = (
    "다음은 여러 주제에 대한 상세한 배경 설명이다. " * 15
    + "이 내용을 참고해서 인공지능의 미래에 대해 설명해줘"
)

results = []
for name, p in [("짧은 prompt", short_prompt), ("중간 prompt", mid_prompt), ("긴 prompt", long_prompt2)]:
    ttft_r, tpot_r, plen_r = measure_ttft_tpot(p, max_new_tokens=20)
    results.append((name, plen_r, ttft_r, tpot_r))
    print(f"{name:10s} | prompt 길이: {plen_r:4d} 토큰 | TTFT: {ttft_r*1000:7.1f} ms | TPOT: {tpot_r*1000:5.1f} ms/tok")

print("\n관찰: prompt 길이가 늘어날수록 TTFT는 뚜렷하게 증가하지만, TPOT는 상대적으로 안정적이다.")
print("-> TTFT는 prefill(=prompt 처리) 비용, TPOT는 decode(=캐시 재사용 스텝) 비용이라는 게 수치로 확인된다.")

## 정리

- Autoregressive 생성은 매 스텝마다 attention을 다시 계산하면 O(N²)로 느려진다 — **KV cache**는 이전 토큰들의 key/value를 저장·재사용해 이를 O(N)에 가깝게 낮춘다.
- 수동 루프(`manual_with_cache` vs `manual_no_cache`) 비교로 캐시가 없을 때의 실제 속도 저하를 직접 확인했다 (0.5B 소형 모델·짧은 시퀀스에서는 배율이 크진 않지만, 시퀀스가 길어질수록 꾸준히 커지는 추세는 명확했다).
- 생성은 **prefill**(prompt 전체 처리, 병렬)과 **decode**(토큰별 순차 생성, 캐시 재사용) 두 단계로 나뉘며, 각각 **TTFT**와 **TPOT**로 측정한다.
- Jetson의 CC8.7 PTX JIT 컴파일 비용은 주로 prefill 시점에 발생하므로, 워밍업 없이 측정한 TTFT는 실제 서빙 지연이 아니라 콜드스타트 아티팩트를 포함할 수 있다 — 벤치마크 전 워밍업은 선택이 아니라 필수다.
- prompt 길이는 TTFT에 영향을 주고, 생성 길이(및 TPOT)는 prompt 길이와 비교적 독립적이다.

**다음 Day 예고**: Day 3~4에서는 양자화(INT8/INT4)로 이 TPOT/메모리 자체를 줄이는 방법을 다룹니다. 오늘 만든 `measure_ttft_tpot` 함수는 양자화 전후 속도 비교에도 그대로 재사용할 수 있습니다.

## 직접 해보기

- `max_new_tokens`를 늘려가며(예: 100, 200) TPOT가 시퀀스가 길어질수록 아주 조금씩 늘어나는지 확인해보세요 (캐시가 커지면서 메모리 접근 비용이 늘어나는 효과).
- `measure_ttft_tpot`에 배치 크기(여러 prompt 동시 처리)를 추가해서 TTFT/TPOT가 어떻게 달라지는지 실험해보세요.
- `torch.cuda.synchronize()`를 일부러 빼고 다시 측정해서 숫자가 어떻게 (잘못) 바뀌는지 확인해보세요 — 왜 동기화가 필수인지 체감할 수 있습니다.